In [2]:
import re
import sys
from pathlib import Path
 
import numpy as np
import pandas as pd

In [ ]:
cwd = Path.cwd()

if (cwd / "src" / "preprocessing" / "model_keys.py").exists():
    ROOT = cwd
elif (cwd.parent / "src" / "preprocessing" / "model_keys.py").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate src/preprocessing/model_keys.py. "
        "Run this notebook from the repository root or notebooks/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing.model_keys import predict_model_key

print("Project root:", ROOT)
print("Model-key rules imported successfully.")


Project root: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT
Model-key rules imported successfully.


In [ ]:
# Check that the required files exist

DATA_DIR = ROOT / "data"
files_to_check = [
    DATA_DIR / "splits" / "validation.csv",
    DATA_DIR / "splits" / "dev_test.csv",
    DATA_DIR / "labels" / "gold_set_Aya.csv",
]

for path in files_to_check:
    print("FOUND:" if path.exists() else "MISSING:", path)

FOUND  e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\splits\validation.csv
FOUND  e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\splits\dev_test.csv


In [16]:
# Check the structure of the files before using them

files_to_inspect = [
    DATA_DIR / "splits" / "validation.csv",
    DATA_DIR / "splits" / "dev_test.csv",
    DATA_DIR / "labels" / "gold_set_Aya.csv",
]

for path in files_to_inspect:
    if not path.exists():
        continue

    df = pd.read_csv(path)

    print(f"\n{path.name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "ad_id" in df.columns:
        print("Unique ad IDs:", df["ad_id"].nunique())

    display(df.head(3))


validation.csv
Shape: (708, 33)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group']
Unique ad IDs: 708


,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,is_feature_phone_hint,price_placeholder,loc_area,loc_city,price_missing,price_suspicious,price_very_high,title_normalized,title_punct_normalized,title_group
0,آبل - آيفون 13 برو ماكس,38000.0,دجله جاردنز، حدائق اكتوبر,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,208962319,legacy,apple,NaN,NaN,NaN,...,False,False,دجله جاردنز,حدائق اكتوبر,False,False,False,ابل - ايفون 13 برو ماكس,ابل ايفون 13 برو ماكس,ابل ايفون 13 برو ماكس
1,زد فولد ٨ للبيع مستعمل ساعتين فقط بحالة الجدي...,100000.0,مدينة المعراج، زهراء المعادى,/ad/%D8%B2%D8%AF-%D9%81%D9%88%D9%84%D8%AF-%D9%...,208704032,legacy,NaN,NaN,NaN,NaN,...,False,False,مدينة المعراج,زهراء المعادى,False,False,False,زد فولد 8 للبيع مستعمل ساعتين فقط بحالة الجدي...,زد فولد 8 للبيع مستعمل ساعتين فقط بحالة الجديد...,زد فولد 8 للبيع مستعمل ساعتين فقط بحالة الجديد...
2,Apple - iPhone 17 Pro Max,83000.0,التجمع الخامس، القاهرة الجديدة,/ad/apple-iphone-17-pro-max-ID208972000.html,208972000,legacy,apple,NaN,NaN,NaN,...,False,False,التجمع الخامس,القاهرة الجديدة,False,False,False,apple - iphone 17 pro max,apple iphone 17 pro max,apple iphone 17 pro max



dev_test.csv
Shape: (575, 33)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group']
Unique ad IDs: 575


,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,is_feature_phone_hint,price_placeholder,loc_area,loc_city,price_missing,price_suspicious,price_very_high,title_normalized,title_punct_normalized,title_group
0,Honor 500 pro,35000.0,6 اكتوبر، الجيزة,/ad/honor-500-pro-ID208029599.html,208029599,legacy,honor,NaN,NaN,NaN,...,False,False,6 اكتوبر,الجيزة,False,False,False,honor 500 pro,honor 500 pro,honor 500 pro
1,Honor Magic V5 dual sim 512/16G White Gold جدي...,63000.0,الوفاء والأمل، مدينة نصر,/ad/honor-magic-v5-dual-sim-51216g-white-gold-...,208918155,legacy,honor,512.0,16.0,L1:slash,...,False,False,الوفاء والأمل,مدينة نصر,False,False,False,honor magic v5 dual sim 512/16g white gold جدي...,honor magic v5 dual sim 512/16g white gold جدي...,honor magic v5 dual sim 512/16g white gold جدي...
2,Honor win rt,30500.0,6 اكتوبر، الجيزة,/ad/honor-win-rt-ID207995757.html,207995757,legacy,honor,NaN,NaN,NaN,...,False,False,6 اكتوبر,الجيزة,False,False,False,honor win rt,honor win rt,honor win rt



gold_set_Aya.csv
Shape: (200, 8)
Columns: ['ad_id', 'title', 'brand', 'gold_model_key', 'review_notes', 'predicted_model_key', 'predicted_match_rule', 'confidence']
Unique ad IDs: 200


,ad_id,title,brand,gold_model_key,review_notes,predicted_model_key,predicted_match_rule,confidence
0,209004543,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple,apple iphone air,العنوان يذكر iPhone Air بالعربية؛ الموديل معرو...,unknown,NaN,NaN
1,209009284,iPhone 12 pro ايفون ١٢ برو,apple,apple iphone 12 pro,NaN,apple iphone 12 pro,apple:ordered_regex,NaN
2,208969053,آبل - آيفون 13,apple,apple iphone 13,NaN,apple iphone 13,apple:ordered_regex,NaN


determinig which data set will be used

In [17]:
validation_df = pd.read_csv(DATA_DIR / "splits" / "validation.csv")
dev_test_df = pd.read_csv(DATA_DIR / "splits" / "dev_test.csv")
gold_df = pd.read_csv(DATA_DIR / "labels" / "gold_set_Aya.csv")

validation_ids = set(validation_df["ad_id"].dropna())
dev_test_ids = set(dev_test_df["ad_id"].dropna())
gold_ids = set(gold_df["ad_id"].dropna())

print("Validation & dev_test overlap:", len(validation_ids & dev_test_ids))
print("Validation & Gold Set overlap:", len(validation_ids & gold_ids))
print("dev_test & Gold Set overlap:", len(dev_test_ids & gold_ids))

Validation & dev_test overlap: 0
Validation & Gold Set overlap: 15
dev_test & Gold Set overlap: 21


In [ ]:
# Prepare candidate ads for manual review + exclude data tthat exists in gold set

review_candidates = validation_df.copy()

review_candidates["ad_id"] = review_candidates["ad_id"].astype(str)
gold_ids = set(gold_df["ad_id"].dropna().astype(str))

before = len(review_candidates)

review_candidates = review_candidates[
    ~review_candidates["ad_id"].isin(gold_ids)
]
review_candidates = review_candidates.drop_duplicates(subset="ad_id").copy()

print("Original validation rows:", before)
print("Remaining candidates:", len(review_candidates))
print("Unique ads:", review_candidates["ad_id"].nunique())

display(review_candidates[["ad_id", "title", "brand"]].head())
#بشيل من الفاليديشن داتا ست ال جولد ست عشان اشتغلنا عليها خلاص

Original validation rows: 708
Remaining candidates: 693
Unique ads: 693


,ad_id,title,brand
0,208962319,آبل - آيفون 13 برو ماكس,apple
1,208704032,زد فولد ٨ للبيع مستعمل ساعتين فقط بحالة الجدي...,NaN
2,208972000,Apple - iPhone 17 Pro Max,apple
3,208459689,آبل - آيفون 17 برو ماكس iphone 17 promax 512,apple
4,208967151,موبايل Samsung S23 Ultra مستعمل (256GB / 12GB ...,samsung


applying model rules on validation dataset 693 rows

In [19]:
# Apply the existing model extraction rules

review_candidates[["predicted_model_key", "match_rule"]] = (
    review_candidates.apply(
        lambda row: predict_model_key(
            row["title"],
            row["brand"],
            with_rule=True
        ),
        axis=1,
        result_type="expand"
    )
) 
display(review_candidates[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule"
]].head(10))

,ad_id,title,brand,predicted_model_key,match_rule
0,208962319,آبل - آيفون 13 برو ماكس,apple,apple iphone 13 pro max,apple:model_pattern_02
1,208704032,زد فولد ٨ للبيع مستعمل ساعتين فقط بحالة الجدي...,NaN,unknown,unknown_brand
2,208972000,Apple - iPhone 17 Pro Max,apple,apple iphone 17 pro max,apple:special_case_09
3,208459689,آبل - آيفون 17 برو ماكس iphone 17 promax 512,apple,apple iphone 17 pro max,apple:special_case_06
4,208967151,موبايل Samsung S23 Ultra مستعمل (256GB / 12GB ...,samsung,samsung s23 ultra,samsung:rule_14
5,203755638,iphone 13 128Gb,apple,apple iphone 13,apple:numeric_pattern
6,209012733,Xiaomi Redmi Note 15 5G,xiaomi,xiaomi redmi note 15 5g,xiaomi:rule_02
7,208417913,سامسونج ايه 23,samsung,samsung a23,samsung:rule_16
8,208718166,Apple - iPhone 17 Pro Max,apple,apple iphone 17 pro max,apple:special_case_09
9,209012573,Apple - iPhone 17,apple,apple iphone 17,apple:numeric_pattern


In [55]:
# Identify ads with unknown predictions for manual review

pred = (
    review_candidates["predicted_model_key"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

rule = (
    review_candidates["match_rule"]
    .fillna("")
    .astype(str)
    .str.strip()
)

is_unknown_prediction = pred.isin(["", "unknown"])
is_unknown_brand = rule.eq("unknown_brand")

is_unknown_known_brand = (
    is_unknown_prediction & ~is_unknown_brand
)

review_candidates["review_bucket"] = "other_rule_review"

review_candidates.loc[
    is_unknown_known_brand,
    "review_bucket"
] = "no_match_known_brand"

review_candidates.loc[
    is_unknown_brand,
    "review_bucket"
] = "unknown_brand"


print("Review bucket counts:")
display(review_candidates["review_bucket"].value_counts())

print("\nUnknown predictions:", is_unknown_prediction.sum())
print("Unknown brand rules:", is_unknown_brand.sum())
print("Unknown predictions with known brand:", is_unknown_known_brand.sum())





Review bucket counts:


review_bucket
other_rule_review       626
unknown_brand            36
no_match_known_brand     31
Name: count, dtype: int64


Unknown predictions: 67
Unknown brand rules: 36
Unknown predictions with known brand: 31


In [56]:
# Select all ads from the first three review categories
review_df = review_candidates[
    review_candidates["review_bucket"].isin([
        "no_match_known_brand",
        "unknown_brand",
    ])
].copy()

# Add columns for manual review
review_df["reviewed_model_key"] = ""
review_df["error_type"] = ""
review_df["notes"] = ""

# Save the review data
output_path = ROOT / "data" / "labels" / "manual_review_candidates.csv"

review_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Review candidates saved to: {output_path}")
print(f"Number of cases: {len(review_df)}")


Review candidates saved to: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\labels\manual_review_candidates.csv
Number of cases: 67


In [57]:
# Check the results
print("Total ads saved:", len(review_df))
print("\nAds by category:")
display(review_df["review_bucket"].value_counts())

print("\nSaved to:", output_path)

display(review_df[
    [
        "ad_id",
        "title",
        "brand",
        "predicted_model_key",
        "match_rule",
        "review_bucket",
        "reviewed_model_key",
        "error_type",
        "notes",
    ]
])


Total ads saved: 67

Ads by category:


review_bucket
unknown_brand           36
no_match_known_brand    31
Name: count, dtype: int64


Saved to: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\labels\manual_review_candidates.csv


,ad_id,title,brand,predicted_model_key,match_rule,review_bucket,reviewed_model_key,error_type,notes
1,208704032,زد فولد ٨ للبيع مستعمل ساعتين فقط بحالة الجدي...,NaN,unknown,unknown_brand,unknown_brand,,,
16,207192878,ناثنج CMF PHONE 1 جديد ٢٥٦ جيجا,NaN,unknown,unknown_brand,unknown_brand,,,
23,209011846,تيلفون موتريلا للبيع اصدار 2024 اندرويد 12,NaN,unknown,unknown_brand,unknown_brand,,,
44,209010223,أخرى.,NaN,unknown,unknown_brand,unknown_brand,,,
61,209008604,S26 Ultra 512 gb Sealed - محلي زيرو ضريبة جديد...,NaN,unknown,unknown_brand,unknown_brand,,,
...,...,...,...,...,...,...,...,...,...
627,208952087,هاتف المهام الشاقة والطوارئ K19 الأصلي,NaN,unknown,unknown_brand,unknown_brand,,,
639,205970876,s24 ultra gray titanium,NaN,unknown,unknown_brand,unknown_brand,,,
659,208948778,شاومي موديل اخر,xiaomi,unknown,xiaomi:no_match,no_match_known_brand,,,
661,208948704,للبدل او للبيع,NaN,unknown,unknown_brand,unknown_brand,,,


# reviewing unknown brands


In [29]:
#reviewing unknown brands
unknown_brand_df = review_candidates[
    review_candidates["review_bucket"] == "unknown_brand"
].copy()

print("Number of unknown-brand cases:", len(unknown_brand_df))

display(
    unknown_brand_df[
        ["ad_id", "title", "brand", "predicted_model_key", "match_rule"]
    ].head(20)
)

Number of unknown-brand cases: 36


,ad_id,title,brand,predicted_model_key,match_rule
1,208704032,زد فولد ٨ للبيع مستعمل ساعتين فقط بحالة الجدي...,NaN,unknown,unknown_brand
16,207192878,ناثنج CMF PHONE 1 جديد ٢٥٦ جيجا,NaN,unknown,unknown_brand
23,209011846,تيلفون موتريلا للبيع اصدار 2024 اندرويد 12,NaN,unknown,unknown_brand
44,209010223,أخرى.,NaN,unknown,unknown_brand
61,209008604,S26 Ultra 512 gb Sealed - محلي زيرو ضريبة جديد...,NaN,unknown,unknown_brand
89,209006923,raelme c65,NaN,unknown,unknown_brand
98,208304653,iqoo neo 9s pro plus snapdragon 8 gen3 12+12 g...,NaN,unknown,unknown_brand
110,209005026,موبايل حالته متفرقش عن الجديد مشاء الله عليه ر...,NaN,unknown,unknown_brand
114,209004830,اي 51,NaN,unknown,unknown_brand
163,209000441,اصغر موبيل اندرويد في العالم استعمال يوم زيرو,NaN,unknown,unknown_brand


In [30]:
no_match_df = review_candidates[
    review_candidates["review_bucket"] == "no_match_known_brand"
].copy()

print("Number of no-match cases:", len(no_match_df))

display(
    no_match_df[
        [
            "ad_id",
            "title",
            "brand",
            "predicted_model_key",
            "match_rule",
        ]
    ]
)

Number of no-match cases: 31


,ad_id,title,brand,predicted_model_key,match_rule
80,209007343,موبايل مستعمل انفنيكس سمارت,infinix,unknown,infinix:no_match
82,209007128,شاومي موديل اخر,xiaomi,unknown,xiaomi:no_match
96,209006078,شاومي موديل اخر,xiaomi,unknown,xiaomi:no_match
103,209005474,شاومي موديل اخر,xiaomi,unknown,xiaomi:no_match
141,209002359,شاومي موديل اخر,xiaomi,unknown,xiaomi:no_match
146,209001983,Infinix Other Model,infinix,unknown,infinix:no_match
171,208999443,OPPO Other Model,oppo,unknown,oppo:no_match
193,208997631,شاومي موديل اخر,xiaomi,unknown,xiaomi:no_match
220,208995279,شاومي موديل اخر,xiaomi,unknown,xiaomi:no_match
251,208992598,شاومي ريد مى,xiaomi,unknown,xiaomi:no_match


# Low confidence cases 

--------------------------------------------------------------------------------------------